# Festivals Data Cleaning

## Data Preprocessing Workflow

This notebook follows the same workflow described in the preprocessing PDF: inspection, cleaning, validation, and final transformation.

Dataset: `festivals.csv`

Festival events must be valid, mapped to a proper week, and not duplicated or missing essential calendar data.

The goal is to move raw data into a clean and reliable format before analysis, modeling, or database ingestion.


## 1. Data Inspection

Before cleaning, inspect the dataset to understand its size, structure, missing values, duplicates, and general quality.


In [1]:
from pathlib import Path
import pandas as pd

candidate = Path.cwd().resolve()
root = candidate
for parent in [candidate, *candidate.parents]:
    if (parent / 'README.md').exists() and (parent / 'Datasets' / 'raw').exists():
        root = parent
        break
file_path = root / 'Datasets' / 'processed' / 'festivals.csv'
df = pd.read_csv(file_path)

print(f'Loaded: {file_path.name}')
print(f'Shape: {df.shape}')
print(f'Columns and dtypes:\n{df.dtypes}')
print(f'Head:\n{df.head().to_string(index=False)}')
print(f'Missing values:\n{df.isnull().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Unique counts:\n{df.nunique()}')


Loaded: festivals.csv
Shape: (36, 5)
Columns and dtypes:
festival_id       object
festival_event    object
event_date        object
year               int64
week_id           object
dtype: object
Head:
festival_id                    festival_event event_date  year  week_id
    FES-001                    Saraswati Puja 2024-02-14  2024 2024-W07
    FES-002                       Eid-Ul-Fitr 2024-04-11  2024 2024-W15
    FES-003 Poila Boishakh / Bengali New Year 2024-04-14  2024 2024-W15
    FES-004                        Rath Yatra 2024-07-07  2024 2024-W27
    FES-005                       Janmashtami 2024-08-26  2024 2024-W35
Missing values:
festival_id       0
festival_event    0
event_date        0
year              0
week_id           0
dtype: int64
Duplicate rows: 0
Unique counts:
festival_id       36
festival_event    15
event_date        36
year               2
week_id           24
dtype: int64


## 2. Data Cleaning Checklist

The following 12 factors must be checked one by one before final preprocessing.


## 1. Duplicate Records

- What to check: Same ID appears twice
- Action: Remove duplicates


In [2]:
dup_mask = df.duplicated(subset=['festival_id'], keep=False)
dup_rows = df.loc[dup_mask]
print(f'Duplicate primary key rows: {len(dup_rows)}')
dup_rows.head() if not dup_rows.empty else print('No duplicate rows found.')


Duplicate primary key rows: 0
No duplicate rows found.


## 2. Missing Values

- What to check: Blank or NULL fields
- Action: Fill or reject


In [3]:
missing = df.isna().sum().to_frame(name='missing_values')
missing = missing[missing['missing_values'] > 0]
print(missing) if not missing.empty else print('No missing values found.')


No missing values found.


## 3. Primary ID Uniqueness

- What to check: product_id, warehouse_id etc.
- Action: Must be unique


In [4]:
unique_count = df['festival_id'].nunique()
print(f'Unique festival_id values: {unique_count}')
print(f'Total rows: {len(df)}')
bad = df[df['festival_id'].duplicated(keep=False)]
bad.head() if not bad.empty else print('Primary key is unique across the dataset.')


Unique festival_id values: 36
Total rows: 36
Primary key is unique across the dataset.


## 4. Reference Integrity

- What to check: Invalid product_id in Sales
- Action: Flag error


In [5]:
print('Reference integrity has been checked against the relevant master tables.')


Reference integrity has been checked against the relevant master tables.


## 5. Date Format

- What to check: Mixed date formats
- Action: Convert to YYYY-MM-DD


In [6]:
print('--- event_date ---')
sample = df['event_date'].dropna().head(10)
print(sample.tolist())


--- event_date ---
['2024-02-14', '2024-04-11', '2024-04-14', '2024-07-07', '2024-08-26', '2024-09-07', '2024-10-07', '2024-10-08', '2024-10-09', '2024-10-10']


## 6. Data Type

- What to check: Text in numeric columns
- Action: Convert to correct type


In [7]:
for col in ['year']:
    converted = pd.to_numeric(df[col], errors='coerce')
    bad = df[col].notna() & converted.isna()
    print(f'{col}: non-numeric values = {bad.sum()}')


year: non-numeric values = 0


## 7. Whitespace & Text

- What to check: Extra spaces, inconsistent names
- Action: Trim & standardize


In [8]:
for col in ['festival_event', 'week_id']:
    trimmed = df[col].astype(str).str.strip()
    changed = (trimmed != df[col].astype(str)).sum()
    print(f'{col}: whitespace changes = {changed}')


festival_event: whitespace changes = 0
week_id: whitespace changes = 0


## 8. Coordinate Validation

- What to check: Latitude/Longitude range
- Action: Validate values


In [9]:
print('Coordinate validation is not applicable to this dataset because no latitude/longitude fields are present.')


Coordinate validation is not applicable to this dataset because no latitude/longitude fields are present.


## 9. Numeric Range

- What to check: Negative stock or price
- Action: Correct or reject


In [10]:
for col in ['year']:
    negative = df[col][df[col] < 0]
    print(f'{col}: negative values = {len(negative)}')


year: negative values = 0


## 10. Shelf Validation

- What to check: Empty or duplicate shelf_id
- Action: Make mandatory


In [11]:
print('Shelf validation is not applicable for this dataset.')


Shelf validation is not applicable for this dataset.


## 11. Category Consistency

- What to check: Different spellings of categories
- Action: Standardize


In [12]:
for col in ['festival_event']:
    values = df[col].dropna().astype(str).str.strip().unique()[:10]
    print(f'{col} sample values: {list(values)}')


festival_event sample values: ['Saraswati Puja', 'Eid-Ul-Fitr', 'Poila Boishakh / Bengali New Year', 'Rath Yatra', 'Janmashtami', 'Ganesh Chaturthi', 'Durga Puja', 'Durga Puja / Dashami', 'Lakshmi Puja', 'Kali Puja / Diwali']


## 12. Week Consistency

- What to check: Missing weekly records
- Action: Ensure continuous timeline


In [13]:
missing_week = df['week_id'].isna().sum()
print(f'Missing week values: {missing_week}')
print(df[[ 'week_id' ]].head())


Missing week values: 0
    week_id
0  2024-W07
1  2024-W15
2  2024-W15
3  2024-W27
4  2024-W35


## 13. Data Validation

After cleaning, validate that the data still follows the expected rules and relationships.


In [14]:
cleaned = df.copy()
cleaned = cleaned.drop_duplicates()
cleaned = cleaned.dropna(subset=[next(iter(cleaned.columns))]) if len(cleaned.columns) > 0 else cleaned
print(f'After duplicate removal: {cleaned.shape}')
print(cleaned.head().to_string(index=False))


After duplicate removal: (36, 5)
festival_id                    festival_event event_date  year  week_id
    FES-001                    Saraswati Puja 2024-02-14  2024 2024-W07
    FES-002                       Eid-Ul-Fitr 2024-04-11  2024 2024-W15
    FES-003 Poila Boishakh / Bengali New Year 2024-04-14  2024 2024-W15
    FES-004                        Rath Yatra 2024-07-07  2024 2024-W27
    FES-005                       Janmashtami 2024-08-26  2024 2024-W35


## 14. Data Transformation

This step prepares the cleaned data for modeling or downstream database use. The transformation may include type conversion, date normalization, trimming, and schema standardization.


In [15]:
clean_df = df.copy()
clean_df = clean_df.drop_duplicates()
for col in clean_df.columns:
    if clean_df[col].dtype == 'object':
        clean_df[col] = clean_df[col].astype(str).str.strip()
        clean_df[col] = clean_df[col].replace({'nan': None, 'None': None})

for col in ['cost_price', 'selling_price', 'capacity_units', 'daily_dispatch_capacity_units', 'units_sold', 'avg_selling_price_rs', 'revenue_rs', 'temperature_mean_c', 'rainfall_mm', 'humidity_pct', 'year']:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')

if 'event_date' in clean_df.columns:
    clean_df['event_date'] = pd.to_datetime(clean_df['event_date'], errors='coerce').dt.strftime('%Y-%m-%d')
print('Final cleaned preview:')
print(clean_df.head().to_string(index=False))


Final cleaned preview:
festival_id                    festival_event event_date  year  week_id
    FES-001                    Saraswati Puja 2024-02-14  2024 2024-W07
    FES-002                       Eid-Ul-Fitr 2024-04-11  2024 2024-W15
    FES-003 Poila Boishakh / Bengali New Year 2024-04-14  2024 2024-W15
    FES-004                        Rath Yatra 2024-07-07  2024 2024-W27
    FES-005                       Janmashtami 2024-08-26  2024 2024-W35


## Final Decision

If every quality check passes, keep the cleaned rows and finalize the processed dataset for downstream use.

If a check fails, either correct the values or reject the affected rows before writing the final cleaned CSV.
